**raw sensitivity Analysis**  

In [1]:
from oreABBR import *; import ORE as ore; import oreUTIL as ou
params  = ore.Parameters()   ; params.fromFile("Input/ore.xml")
mainOBJ = ore.OREApp(params) ; mainOBJ.run()
ou.chkERR(mainOBJ)
print('\n asof:',mainOBJ.getInputs().asof()); mainOBJ.getReportNames()

Run time:0.17 sec, Errors:0

 asof: August 19th, 2022


('dividends',
 'fixings',
 'marketdata',
 'pricingstats',
 'runtimes',
 'sensitivity',
 'sensitivity_config',
 'sensitivity_scenario',
 'todaysmarketcalibration',
 'todaysmarketcalibration_cashflows')

In [2]:
snOBJ = mainOBJ.getReport("sensitivity")
dfSN  = ou.rOBJxdf(snOBJ); print('行,列:',dfSN.shape); display(dfSN[5:])

行,列: (8, 14)


,TradeId,IsPar,Factor_1,ShiftSize_1,Factor_2,ShiftSize_2,Currency,Base NPV,Delta,Gamma,Currency(Trade),Base NPV(Trade),Delta(Trade),Gamma(Trade)
5,OIS-2y,false,IndexCurve/JPY-TONAR/2/1Y,0.0001,,0.0,JPY,-230413.271223,0.234047,0.195638,JPY,-230413.271223,0.234047,0.195638
6,OIS-2y,false,IndexCurve/JPY-TONAR/3/2Y,0.0001,,0.0,JPY,-230413.271223,1980.707041,0.383724,JPY,-230413.271223,1980.707041,0.383724
7,OIS-2y,false,IndexCurve/JPY-TONAR/4/3Y,0.0001,,0.0,JPY,-230413.271223,32.902144,0.000108,JPY,-230413.271223,32.902144,0.000108


In [3]:
oeSENSI = ['TradeId','Factor_1','Currency','Base NPV','ShiftSize_1','Delta','Gamma']
dfDSP(dfSN[oeSENSI], fm=fmO )
print( f'Total Delta:{(dfSN.Delta).sum():,.2f}   '
       f'Total Gamma:{(dfSN.Gamma).sum():,.2f}'   )

,TradeId,Factor_1,Currency,Base NPV,ShiftSize_1,Delta,Gamma
0,OIS-2y,DiscountCurve/JPY/2/1Y,JPY,"-230,413.27",0.0100%,11.67,-0.00
1,OIS-2y,DiscountCurve/JPY/3/2Y,JPY,"-230,413.27",0.0100%,22.27,-0.00
2,OIS-2y,DiscountCurve/JPY/4/3Y,JPY,"-230,413.27",0.0100%,0.74,-0.00
3,OIS-2y,IndexCurve/JPY-TONAR/0/1D,JPY,"-230,413.27",0.0100%,-2.69,0.00
4,OIS-2y,IndexCurve/JPY-TONAR/1/6M,JPY,"-230,413.27",0.0100%,-8.26,0.00
5,OIS-2y,IndexCurve/JPY-TONAR/2/1Y,JPY,"-230,413.27",0.0100%,0.23,0.20
6,OIS-2y,IndexCurve/JPY-TONAR/3/2Y,JPY,"-230,413.27",0.0100%,"1,980.71",0.38
7,OIS-2y,IndexCurve/JPY-TONAR/4/3Y,JPY,"-230,413.27",0.0100%,32.90,0.00


Total Delta:2,037.57   Total Gamma:0.57


In [4]:
print( "A/B/C".split("/")    ,
    	 "A/B/C".rsplit("/",1) ) 									  # 注: n=1ではエラー

['A', 'B', 'C'] ['A/B', 'C']


In [5]:
dfSN.Factor_1.str.rsplit("/",n=1) 								# 注: 1ではエラー

0       [DiscountCurve/JPY/2, 1Y]
1       [DiscountCurve/JPY/3, 2Y]
2       [DiscountCurve/JPY/4, 3Y]
3    [IndexCurve/JPY-TONAR/0, 1D]
4    [IndexCurve/JPY-TONAR/1, 6M]
5    [IndexCurve/JPY-TONAR/2, 1Y]
6    [IndexCurve/JPY-TONAR/3, 2Y]
7    [IndexCurve/JPY-TONAR/4, 3Y]
Name: Factor_1, dtype: object

In [6]:
type(dfSN.Factor_1.str.rsplit("/",n=1))

pandas.core.series.Series

In [7]:
dfSN.Factor_1.str.rsplit("/", n=1).str[-1]

0    1Y
1    2Y
2    3Y
3    1D
4    6M
5    1Y
6    2Y
7    3Y
Name: Factor_1, dtype: object

In [8]:
dfSN["Tenor"] = dfSN.Factor_1.str.rsplit("/", n=1).str[-1]
oeSENSI[1] = 'Tenor'; dfDSP(dfSN[oeSENSI],fm=fmO)

,TradeId,Tenor,Currency,Base NPV,ShiftSize_1,Delta,Gamma
0,OIS-2y,1Y,JPY,"-230,413.27",0.0100%,11.67,-0.00
1,OIS-2y,2Y,JPY,"-230,413.27",0.0100%,22.27,-0.00
2,OIS-2y,3Y,JPY,"-230,413.27",0.0100%,0.74,-0.00
3,OIS-2y,1D,JPY,"-230,413.27",0.0100%,-2.69,0.00
4,OIS-2y,6M,JPY,"-230,413.27",0.0100%,-8.26,0.00
5,OIS-2y,1Y,JPY,"-230,413.27",0.0100%,0.23,0.20
6,OIS-2y,2Y,JPY,"-230,413.27",0.0100%,"1,980.71",0.38
7,OIS-2y,3Y,JPY,"-230,413.27",0.0100%,32.90,0.00


In [9]:
colCAL = {"Delta":"sum", "Gamma":"sum"}
dfGPS  = (dfSN.groupby(['Tenor'], as_index=False).agg(colCAL))
dfDSP(dfGPS,fm=fmO)

,Tenor,Delta,Gamma
0,1D,-2.69,0.00
1,1Y,11.91,0.19
2,2Y,"2,002.98",0.38
3,3Y,33.64,0.00
4,6M,-8.26,0.00


In [10]:
dfGPS = dfGPS.loc[[0,4,1,2,3]].reset_index(drop=True);dfDSP(dfGPS,fm=fmO)
print( f'Total Delta:{(dfSN.Delta).sum():,.2f}   '
       f'Total Gamma:{(dfSN.Gamma).sum():,.2f}')

,Tenor,Delta,Gamma
0,1D,-2.69,0.00
1,6M,-8.26,0.00
2,1Y,11.91,0.19
3,2Y,"2,002.98",0.38
4,3Y,33.64,0.00


Total Delta:2,037.57   Total Gamma:0.57


In [11]:
dfGPS = ou.calcSingleGPS(dfSN); dfDSP(dfGPS,fm=fmO)

,Tenor,Delta,Gamma
0,1D,-2.69,0.00
1,6M,-8.26,0.00
2,1Y,11.91,0.19
3,2Y,"2,002.98",0.38
4,3Y,33.64,0.00
